# Swarm Demo — Multi-Agent Log Analysis with Memory Pointer Pattern

Based on: [Solving Context Window Overflow in AI Agents](https://arxiv.org/html/2511.22729v1) — IBM Research, 2024

## The Problem

In a multi-agent workflow, agents need to share large datasets. Passing 145KB of logs as a message between agents overflows each agent's context window. The data needs to exist somewhere all agents can access — without any of them holding it in their context.

## The Solution: `invocation_state`

`tool_context.invocation_state` is the Strands API for sharing data across agents in a swarm. Tools write large data there and return only a pointer key. Any agent in the swarm can read that key from `invocation_state` — the raw data never enters any LLM context window.

```
agent.state         → scoped to one agent (single-agent pattern)
invocation_state    → shared across all agents in the swarm (multi-agent pattern)
```

## The Tools

Five tools in `swarm_demo.py` built on `ToolContext.invocation_state`:

| Tool | What it does | Writes to invocation_state | Reads from invocation_state |
|------|-------------|---------------------------|------------------------------|
| `fetch_application_logs(app_name, hours)` | Generates log events, stores them | `"logs-{app_name}"` | — |
| `analyze_error_patterns(logs_pointer)` | Counts errors by service | `"error_analysis"` | `logs_pointer` |
| `detect_latency_anomalies(logs_pointer)` | Calculates p95 latency | `"latency_analysis"` | `logs_pointer` |
| `generate_incident_report()` | Combines both analyses | — | `"error_analysis"`, `"latency_analysis"` |
| `get_error_details(logs_pointer, service)` | Drills into errors for one service | — | `logs_pointer` |

## Architecture

```
┌───────────────────────────────────────────────────────────────────┐
│                        invocation_state                            │
│  "logs-payment-service"  →  [600 events, 145KB]                   │
│  "error_analysis"        →  {total_errors, by_service, ...}       │
│  "latency_analysis"      →  {p95_latency_ms, anomalies_count, ...}│
└───────┬──────────────────────┬────────────────────────────────────┘
   write│               read+write│                         read│
        ▼                         ▼                              ▼
┌──────────────┐   ┌──────────────────────┐   ┌──────────────────────┐
│  Collector   │──►│       Analyzer       │──►│      Reporter        │
│ fetch_logs() │   │ analyze_errors()     │   │ generate_report()    │
│              │   │ detect_anomalies()   │   │                      │
└──────────────┘   └──────────────────────┘   └──────────────────────┘
```

## What We Test

| Step | What it shows |
|------|---------------|
| 1 — Run the Swarm | Collector → Analyzer → Reporter pipeline, data flowing via invocation_state |
| 2 — Follow-up investigation | Investigator agent reuses stored data after swarm completes — no re-fetch |

In [ ]:
import os
os.environ['OTEL_SDK_DISABLED'] = 'true'

from dotenv import load_dotenv
load_dotenv()

from swarm_demo_tools import (
    fetch_application_logs, analyze_error_patterns,
    detect_latency_anomalies, generate_incident_report, get_error_details,
    collector, analyzer, reporter, swarm, MODEL,
)

print("✅ Swarm ready: collector → analyzer → reporter")

---
## Step 1 — Run the Swarm

One request triggers the full Collector → Analyzer → Reporter pipeline. While it runs, notice:

- Each agent calls only its own tools — the Collector never calls `generate_incident_report`, the Reporter never calls `fetch_logs`
- The LLM in each agent sees pointer strings like `"logs-payment-service"`, not 145KB of raw JSON
- `node_history` in the result shows the handoff order

> Note: 6 hours × 100 events/hour = 600 log events. This will take ~20–30 seconds.

In [ ]:
print("⏳ Running swarm (~20–30s)...\n")

result = swarm("Fetch 6 hours of logs for payment-service, analyze errors and latency, then generate an incident report.")

print(f"\nStatus:     {result.status}")
print(f"Agents:     {' → '.join(n.node_id for n in result.node_history)}")
print(f"Time:       {result.execution_time}ms")

---
## Step 2 — Follow-up Investigation

The swarm completed — but `invocation_state` still holds everything: the 145KB of logs, the error analysis, and the latency analysis. An investigator agent can now ask follow-up questions **without re-fetching any data**.

This is the key advantage over a single-agent approach: in a traditional system, each follow-up question would require re-fetching 145KB of logs. With `invocation_state`, the data persists for the lifetime of the workflow and any agent can access it at any time.

In [ ]:
from strands import Agent

investigator = Agent(
    name="investigator",
    system_prompt="You investigate incidents. The logs pointer is 'logs-payment-service'. Use get_error_details to drill into specific services.",
    tools=[get_error_details, analyze_error_patterns],
    model=MODEL,
)

print("👤 Turn 1: Which service had the most errors?\n")
investigator("Based on the error analysis in shared state, which service had the most errors? The logs are at 'logs-payment-service'")

print("\n👤 Turn 2: Show me the actual error logs for that service\n")
investigator("Show me 3 detailed error logs for the service with the most errors")

print("\n👤 Turn 3: What status codes are those errors?\n")
investigator("What HTTP status codes are those errors returning?")

print("\n📦 Data persisted in invocation_state throughout — never re-fetched")

---
## Key Takeaways

1. **Swarm handles coordination** — collector → analyzer → reporter with autonomous handoffs
2. **`invocation_state` for multi-agent data** — the official Strands API for sharing data across agents in a swarm
3. **Large data stays out of context** — 145KB+ of logs in `invocation_state`, only pointers in LLM context
4. **Data persists after swarm completes** — follow-up investigation reuses the same stored data without re-fetching
5. **Same ToolContext API** — single-agent uses `agent.state`, multi-agent uses `invocation_state`, both via `ToolContext`

## References

- [Strands Swarm](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/swarm/) — Multi-agent orchestration
- [Shared State Across Multi-Agent Patterns](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/multi-agent/multi-agent-patterns/) — invocation_state for data sharing
- [Strands ToolContext](https://strandsagents.com/latest/documentation/docs/user-guide/concepts/tools/creating-custom-tools/) — Accessing agent.state and invocation_state
- [Solving Context Window Overflow](https://arxiv.org/html/2511.22729v1) — IBM Research
- [Towards Effective GenAI Multi-Agent Collaboration](https://arxiv.org/pdf/2412.05449) — Amazon, payload referencing
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)